In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Import the necessary function from threadpoolctl
from threadpoolctl import threadpool_limits

# Load and preprocess the data (same as before)
data_path = "C:/Users/olufe/projects/Journal/dataset/HomeA_unsupervised_real_combined_shuffled.csv"
data = pd.read_csv(data_path)
X = data.drop(columns=["Label"])
y = data["Label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [40]:
# Build the Deep Autoencoder

def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder

# Assuming the input dimension is the number of features in the dataset
input_dim = X_train_scaled.shape[1]
autoencoder = build_autoencoder(input_dim)

In [41]:
# Train the autoencoder (same as before)
autoencoder = build_autoencoder(input_dim)
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])

Epoch 1/50
592/592 [==============================] - 6s 8ms/step - loss: 0.7206 - val_loss: 0.5951
Epoch 2/50
592/592 [==============================] - 5s 8ms/step - loss: 0.5650 - val_loss: 0.5303
Epoch 3/50
592/592 [==============================] - 5s 8ms/step - loss: 0.5191 - val_loss: 0.5013
Epoch 4/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4965 - val_loss: 0.4834
Epoch 5/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4822 - val_loss: 0.4749
Epoch 6/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4714 - val_loss: 0.4623
Epoch 7/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4618 - val_loss: 0.4563
Epoch 8/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4556 - val_loss: 0.4524
Epoch 9/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4504 - val_loss: 0.4498
Epoch 10/50
592/592 [==============================] - 5s 8ms/step - loss: 0.4466 - val_loss: 0.4440

In [42]:
# Define the number of clusters for k-means
num_clusters = 5

# Initialize k-means with data points and get cluster memberships
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
X_train_clusters = kmeans.fit_predict(X_train_scaled)

# Create separate autoencoders for each cluster
cluster_autoencoders = []
for cluster_id in range(num_clusters):
    cluster_samples = X_train_scaled[X_train_clusters == cluster_id]
    cluster_autoencoder = build_autoencoder(input_dim)
    cluster_autoencoder.fit(cluster_samples, cluster_samples, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])
    cluster_autoencoders.append(cluster_autoencoder)

Epoch 1/50
124/124 [==============================] - 2s 9ms/step - loss: 0.9104 - val_loss: 0.7500
Epoch 2/50
124/124 [==============================] - 1s 8ms/step - loss: 0.6754 - val_loss: 0.6251
Epoch 3/50
124/124 [==============================] - 1s 8ms/step - loss: 0.5982 - val_loss: 0.5698
Epoch 4/50
124/124 [==============================] - 1s 8ms/step - loss: 0.5489 - val_loss: 0.5298
Epoch 5/50
124/124 [==============================] - 1s 8ms/step - loss: 0.5137 - val_loss: 0.5037
Epoch 6/50
124/124 [==============================] - 1s 7ms/step - loss: 0.4879 - val_loss: 0.4813
Epoch 7/50
124/124 [==============================] - 1s 8ms/step - loss: 0.4692 - val_loss: 0.4639
Epoch 8/50
124/124 [==============================] - 1s 8ms/step - loss: 0.4540 - val_loss: 0.4555
Epoch 9/50
124/124 [==============================] - 1s 8ms/step - loss: 0.4430 - val_loss: 0.4450
Epoch 10/50
124/124 [==============================] - 1s 8ms/step - loss: 0.4347 - val_loss: 0.4389

Epoch 37/50
67/67 [==============================] - 1s 8ms/step - loss: 0.3481 - val_loss: 0.3632
Epoch 38/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3464 - val_loss: 0.3633
Epoch 39/50
67/67 [==============================] - 1s 8ms/step - loss: 0.3454 - val_loss: 0.3603
Epoch 40/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3438 - val_loss: 0.3588
Epoch 41/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3425 - val_loss: 0.3583
Epoch 42/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3421 - val_loss: 0.3603
Epoch 43/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3406 - val_loss: 0.3572
Epoch 44/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3390 - val_loss: 0.3563
Epoch 45/50
67/67 [==============================] - 0s 7ms/step - loss: 0.3391 - val_loss: 0.3564
Epoch 46/50
67/67 [==============================] - 1s 8ms/step - loss: 0.3381 - val_loss: 0.3565
Epoch 47/5

288/288 [==============================] - 2s 6ms/step - loss: 0.1671 - val_loss: 0.1695
Epoch 20/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1665 - val_loss: 0.1689
Epoch 21/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1659 - val_loss: 0.1687
Epoch 22/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1655 - val_loss: 0.1688
Epoch 23/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1647 - val_loss: 0.1673
Epoch 24/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1644 - val_loss: 0.1673
Epoch 25/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1641 - val_loss: 0.1673
Epoch 26/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1635 - val_loss: 0.1662
Epoch 27/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1632 - val_loss: 0.1659
Epoch 28/50
288/288 [==============================] - 2s 6ms/step - loss: 0.1628 - val_loss: 0.1664
Ep

In [43]:
# Perform self-training iterations
num_self_training_iterations = 3
for iteration in range(num_self_training_iterations):
    # Obtain reconstruction errors for all samples using the ensemble
    ensemble_reconstruction_errors = np.zeros(X_train_scaled.shape[0])
    for cluster_id, cluster_autoencoder in enumerate(cluster_autoencoders):
        cluster_samples = X_train_scaled[X_train_clusters == cluster_id]
        reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
        ensemble_reconstruction_errors[X_train_clusters == cluster_id] = reconstruction_errors

    # Identify unlabeled anomalies and add them to the training set
    threshold = np.percentile(ensemble_reconstruction_errors, 95)
    unlabeled_anomalies_indices = np.where(ensemble_reconstruction_errors > threshold)[0]
    if len(unlabeled_anomalies_indices) == 0:
        break
    X_train_scaled = np.concatenate((X_train_scaled, X_train_scaled[unlabeled_anomalies_indices]), axis=0)

    # Update k-means clustering with the updated training set
    X_train_clusters = kmeans.fit_predict(X_train_scaled)

    # Update cluster autoencoders with the updated training set
    for cluster_id in range(num_clusters):
        cluster_samples = X_train_scaled[X_train_clusters == cluster_id]
        cluster_autoencoder = build_autoencoder(input_dim)
        cluster_autoencoder.fit(cluster_samples, cluster_samples, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])
        cluster_autoencoders[cluster_id] = cluster_autoencoder

133/133 [==============================] - 0s 2ms/step
Epoch 1/50
305/305 [==============================] - 3s 6ms/step - loss: 0.3265 - val_loss: 0.3561
Epoch 2/50
305/305 [==============================] - 2s 6ms/step - loss: 0.2460 - val_loss: 0.3182
Epoch 3/50
305/305 [==============================] - 2s 6ms/step - loss: 0.2181 - val_loss: 0.2993
Epoch 4/50
305/305 [==============================] - 2s 6ms/step - loss: 0.2043 - val_loss: 0.2886
Epoch 5/50
305/305 [==============================] - 2s 6ms/step - loss: 0.1951 - val_loss: 0.2824
Epoch 6/50
305/305 [==============================] - 2s 6ms/step - loss: 0.1889 - val_loss: 0.2768
Epoch 7/50
305/305 [==============================] - 2s 6ms/step - loss: 0.1847 - val_loss: 0.2724
Epoch 8/50
305/305 [==============================] - 2s 6ms/step - loss: 0.1812 - val_loss: 0.2694
Epoch 9/50
305/305 [==============================] - 2s 6ms/step - loss: 0.1787 - val_loss: 0.2668
Epoch 10/50
305/305 [========================

82/82 [==============================] - 1s 8ms/step - loss: 1.2025 - val_loss: 1.5969
Epoch 4/50
82/82 [==============================] - 1s 8ms/step - loss: 1.1087 - val_loss: 1.5121
Epoch 5/50
82/82 [==============================] - 1s 8ms/step - loss: 1.0461 - val_loss: 1.4482
Epoch 6/50
82/82 [==============================] - 1s 8ms/step - loss: 1.0058 - val_loss: 1.4065
Epoch 7/50
82/82 [==============================] - 1s 8ms/step - loss: 0.9731 - val_loss: 1.3575
Epoch 8/50
82/82 [==============================] - 1s 7ms/step - loss: 0.9476 - val_loss: 1.3185
Epoch 9/50
82/82 [==============================] - 1s 8ms/step - loss: 0.9236 - val_loss: 1.2929
Epoch 10/50
82/82 [==============================] - 1s 8ms/step - loss: 0.9039 - val_loss: 1.2647
Epoch 11/50
82/82 [==============================] - 1s 8ms/step - loss: 0.8861 - val_loss: 1.2398
Epoch 12/50
82/82 [==============================] - 1s 8ms/step - loss: 0.8702 - val_loss: 1.2199
Epoch 13/50
82/82 [=========

156/156 [==============================] - 1s 7ms/step - loss: 0.3885 - val_loss: 0.6281
Epoch 36/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3871 - val_loss: 0.6275
Epoch 37/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3862 - val_loss: 0.6255
Epoch 38/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3854 - val_loss: 0.6238
Epoch 39/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3846 - val_loss: 0.6240
Epoch 40/50
156/156 [==============================] - 1s 6ms/step - loss: 0.3839 - val_loss: 0.6260
Epoch 41/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3829 - val_loss: 0.6232
Epoch 42/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3819 - val_loss: 0.6205
Epoch 43/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3813 - val_loss: 0.6213
Epoch 44/50
156/156 [==============================] - 1s 7ms/step - loss: 0.3808 - val_loss: 0.6194
Ep

304/304 [==============================] - 2s 6ms/step - loss: 0.1678 - val_loss: 0.3994
Epoch 18/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1669 - val_loss: 0.3985
Epoch 19/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1663 - val_loss: 0.3977
Epoch 20/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1658 - val_loss: 0.3980
Epoch 21/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1652 - val_loss: 0.3952
Epoch 22/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1648 - val_loss: 0.3958
Epoch 23/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1640 - val_loss: 0.3927
Epoch 24/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1637 - val_loss: 0.3924
Epoch 25/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1632 - val_loss: 0.3928
Epoch 26/50
304/304 [==============================] - 2s 6ms/step - loss: 0.1629 - val_loss: 0.3919
Ep

123/123 [==============================] - 1s 6ms/step - loss: 0.4390 - val_loss: 0.9542
Epoch 12/50
123/123 [==============================] - 1s 6ms/step - loss: 0.4330 - val_loss: 0.9485
Epoch 13/50
123/123 [==============================] - 1s 5ms/step - loss: 0.4256 - val_loss: 0.9388
Epoch 14/50
123/123 [==============================] - 1s 6ms/step - loss: 0.4215 - val_loss: 0.9309
Epoch 15/50
123/123 [==============================] - 1s 6ms/step - loss: 0.4154 - val_loss: 0.9249
Epoch 16/50
123/123 [==============================] - 1s 5ms/step - loss: 0.4115 - val_loss: 0.9164
Epoch 17/50
123/123 [==============================] - 1s 6ms/step - loss: 0.4083 - val_loss: 0.9167
Epoch 18/50
123/123 [==============================] - 1s 6ms/step - loss: 0.4037 - val_loss: 0.9150
Epoch 19/50
123/123 [==============================] - 1s 6ms/step - loss: 0.4022 - val_loss: 0.9086
Epoch 20/50
123/123 [==============================] - 1s 5ms/step - loss: 0.3985 - val_loss: 0.9093
Ep

99/99 [==============================] - 1s 5ms/step - loss: 0.7853 - val_loss: 1.1699
Epoch 3/50
99/99 [==============================] - 1s 5ms/step - loss: 0.6832 - val_loss: 1.0392
Epoch 4/50
99/99 [==============================] - 1s 5ms/step - loss: 0.6274 - val_loss: 0.9823
Epoch 5/50
99/99 [==============================] - 1s 6ms/step - loss: 0.5950 - val_loss: 0.9440
Epoch 6/50
99/99 [==============================] - 1s 6ms/step - loss: 0.5720 - val_loss: 0.9211
Epoch 7/50
99/99 [==============================] - 1s 5ms/step - loss: 0.5530 - val_loss: 0.9050
Epoch 8/50
99/99 [==============================] - 1s 5ms/step - loss: 0.5357 - val_loss: 0.8780
Epoch 9/50
99/99 [==============================] - 1s 5ms/step - loss: 0.5218 - val_loss: 0.8509
Epoch 10/50
99/99 [==============================] - 1s 5ms/step - loss: 0.5088 - val_loss: 0.8306
Epoch 11/50
99/99 [==============================] - 1s 5ms/step - loss: 0.4977 - val_loss: 0.8061
Epoch 12/50
99/99 [==========

156/156 [==============================] - 1s 5ms/step - loss: 0.4046 - val_loss: 0.9618
Epoch 39/50
156/156 [==============================] - 1s 5ms/step - loss: 0.4038 - val_loss: 0.9571
Epoch 40/50
156/156 [==============================] - 1s 6ms/step - loss: 0.4031 - val_loss: 0.9588
Epoch 41/50
156/156 [==============================] - 1s 6ms/step - loss: 0.4020 - val_loss: 0.9518
Epoch 42/50
156/156 [==============================] - 1s 5ms/step - loss: 0.4011 - val_loss: 0.9518
Epoch 43/50
156/156 [==============================] - 1s 5ms/step - loss: 0.4008 - val_loss: 0.9490
Epoch 44/50
156/156 [==============================] - 1s 5ms/step - loss: 0.4000 - val_loss: 0.9466
Epoch 45/50
156/156 [==============================] - 1s 5ms/step - loss: 0.3997 - val_loss: 0.9449
Epoch 46/50
156/156 [==============================] - 1s 5ms/step - loss: 0.3980 - val_loss: 0.9464
Epoch 47/50
156/156 [==============================] - 1s 5ms/step - loss: 0.3984 - val_loss: 0.9406
Ep

Epoch 25/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5840 - val_loss: 0.8239
Epoch 26/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5816 - val_loss: 0.8215
Epoch 27/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5783 - val_loss: 0.8126
Epoch 28/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5749 - val_loss: 0.8068
Epoch 29/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5715 - val_loss: 0.7999
Epoch 30/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5689 - val_loss: 0.7945
Epoch 31/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5669 - val_loss: 0.7905
Epoch 32/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5631 - val_loss: 0.7958
Epoch 33/50
81/81 [==============================] - 1s 10ms/step - loss: 0.5609 - val_loss: 0.7862
Epoch 34/50
81/81 [==============================] - 1s 9ms/step - loss: 0.5586 - val_loss: 0.7817
Epoch 35/

Epoch 7/50
10/10 [==============================] - 0s 11ms/step - loss: 1.3249 - val_loss: 1.2701
Epoch 8/50
10/10 [==============================] - 0s 13ms/step - loss: 1.2291 - val_loss: 1.1736
Epoch 9/50
10/10 [==============================] - 0s 12ms/step - loss: 1.1291 - val_loss: 1.0833
Epoch 10/50
10/10 [==============================] - 0s 11ms/step - loss: 1.0384 - val_loss: 1.0186
Epoch 11/50
10/10 [==============================] - 0s 12ms/step - loss: 0.9826 - val_loss: 0.9822
Epoch 12/50
10/10 [==============================] - 0s 12ms/step - loss: 0.9481 - val_loss: 0.9508
Epoch 13/50
10/10 [==============================] - 0s 11ms/step - loss: 0.9183 - val_loss: 0.9366
Epoch 14/50
10/10 [==============================] - 0s 11ms/step - loss: 0.8956 - val_loss: 0.9136
Epoch 15/50
10/10 [==============================] - 0s 12ms/step - loss: 0.8745 - val_loss: 0.8911
Epoch 16/50
10/10 [==============================] - 0s 11ms/step - loss: 0.8519 - val_loss: 0.8701
Epo

In [44]:
# Now let's evaluate the final model on the test set
y_test_pred = []
# Initialize k-means for test data
kmeans_test = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to test data and get cluster memberships
X_test_clusters = kmeans_test.fit_predict(X_test_scaled)

for cluster_id in range(num_clusters):
    # Get the indices of the cluster samples in the test set
    cluster_indices = np.where(X_test_clusters == cluster_id)[0]

    # Check if there are cluster samples in the test set
    if len(cluster_indices) == 0:
        continue  # Skip this cluster if there are no samples in the test set

    cluster_samples = X_test_scaled[cluster_indices]
    cluster_autoencoder = cluster_autoencoders[cluster_id]
    reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
    y_test_pred.extend([1 if err > threshold else 0 for err in reconstruction_errors])

60/60 [==============================] - 0s 3ms/step


In [45]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Convert y_test_pred to numpy array
y_test_pred = np.array(y_test_pred)

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
tnr = tn / (tn + fp)
fpr = fp / (tn + fp)
fnr = fn / (fn + tp)
f1 = f1_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_pred)

# Print the performance metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("TNR:", tnr)
print("FPR:", fpr)
print("FNR:", fnr)
print("F1-score:", f1)
print("AUC:", auc)

Accuracy: 0.7009132420091324
Precision: 0.16857688634192933
Recall: 0.2010250569476082
TNR: 0.8011649154865236
FPR: 0.19883508451347648
FNR: 0.7989749430523918
F1-score: 0.18337662337662336
AUC: 0.5010949862170658


In [46]:
#Evaluating Semi-Supervised Ensemble model

In [9]:
# Now let's evaluate the final model on the test set
y_test_pred_semi_supervised = []
# Initialize k-means for test data
kmeans_test = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to test data and get cluster memberships
X_test_clusters = kmeans_test.fit_predict(X_test_scaled)

for cluster_id in range(num_clusters):
    # Get the indices of the cluster samples in the test set
    cluster_indices = np.where(X_test_clusters == cluster_id)[0]

    # Check if there are cluster samples in the test set
    if len(cluster_indices) == 0:
        continue  # Skip this cluster if there are no samples in the test set

    cluster_samples = X_test_scaled[cluster_indices]
    cluster_autoencoder = cluster_autoencoders[cluster_id]
    reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
    y_test_pred_semi_supervised.extend([1 if err > threshold else 0 for err in reconstruction_errors])

60/60 [==============================] - 0s 3ms/step


In [10]:
# Convert y_test_pred to numpy array
y_test_pred_semi_supervised = np.array(y_test_pred_semi_supervised)

# Calculate performance metrics
accuracy_semi_supervised = accuracy_score(y_test, y_test_pred_semi_supervised)
precision_semi_supervised = precision_score(y_test, y_test_pred_semi_supervised)
recall_semi_supervised = recall_score(y_test, y_test_pred_semi_supervised)
conf_matrix_semi_supervised = confusion_matrix(y_test, y_test_pred_semi_supervised)
tn_semi_supervised, fp_semi_supervised, fn_semi_supervised, tp_semi_supervised = conf_matrix_semi_supervised.ravel()
tnr_semi_supervised = tn_semi_supervised / (tn_semi_supervised + fp_semi_supervised)
fpr_semi_supervised = fp_semi_supervised / (tn_semi_supervised + fp_semi_supervised)
fnr_semi_supervised = fn_semi_supervised / (fn_semi_supervised + tp_semi_supervised)
f1_semi_supervised = 2 * (precision_semi_supervised * recall_semi_supervised) / (precision_semi_supervised + recall_semi_supervised)
auc_semi_supervised = roc_auc_score(y_test, y_test_pred_semi_supervised)

# Print the performance metrics for Semi-Supervised Ensemble method
print("Semi-Supervised Ensemble Method")
print("Accuracy:", accuracy_semi_supervised)
print("Precision:", precision_semi_supervised)
print("Recall:", recall_semi_supervised)
print("TNR:", tnr_semi_supervised)
print("FPR:", fpr_semi_supervised)
print("FNR:", fnr_semi_supervised)
print("F1-score:", f1_semi_supervised)
print("AUC:", auc_semi_supervised)

Semi-Supervised Ensemble Method
Accuracy: 0.6388888888888888
Precision: 0.16469428007889547
Recall: 0.2853075170842825
TNR: 0.7097989949748744
FPR: 0.29020100502512564
FNR: 0.7146924829157175
F1-score: 0.20883701542309296
AUC: 0.49755325602957845


In [11]:
#Now let's implement the Self-Training method:

In [12]:
# Initialize k-means for self-training
kmeans_self_training = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to the entire training data and get cluster memberships
X_clusters_self_training = kmeans_self_training.fit_predict(X_train_scaled)

# Perform self-training iterations (same as before)
# ...

# Now let's evaluate the final model on the test set
y_test_pred_self_training = []
# Initialize k-means for test data
kmeans_test = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to test data and get cluster memberships
X_test_clusters = kmeans_test.fit_predict(X_test_scaled)

for cluster_id in range(num_clusters):
    # Get the indices of the cluster samples in the test set
    cluster_indices = np.where(X_test_clusters == cluster_id)[0]

    # Check if there are cluster samples in the test set
    if len(cluster_indices) == 0:
        continue  # Skip this cluster if there are no samples in the test set

    cluster_samples = X_test_scaled[cluster_indices]
    cluster_autoencoder = cluster_autoencoders[cluster_id]
    reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
    y_test_pred_self_training.extend([1 if err > threshold else 0 for err in reconstruction_errors])

# Convert y_test_pred to numpy array
y_test_pred_self_training = np.array(y_test_pred_self_training)

# Calculate performance metrics
accuracy_self_training = accuracy_score(y_test, y_test_pred_self_training)
precision_self_training = precision_score(y_test, y_test_pred_self_training)
recall_self_training = recall_score(y_test, y_test_pred_self_training)
conf_matrix_self_training = confusion_matrix(y_test, y_test_pred_self_training)
tn_self_training, fp_self_training, fn_self_training, tp_self_training = conf_matrix_self_training.ravel()
tnr_self_training = tn_self_training / (tn_self_training + fp_self_training)
fpr_self_training = fp_self_training / (tn_self_training + fp_self_training)
fnr_self_training = fn_self_training / (fn_self_training + tp_self_training)
f1_self_training = 2 * (precision_self_training * recall_self_training) / (precision_self_training + recall_self_training)
auc_self_training = roc_auc_score(y_test, y_test_pred_self_training)

# Print the performance metrics for Self-Training method
print("Self-Training Method")
print("Accuracy:", accuracy_self_training)
print("Precision:", precision_self_training)
print("Recall:", recall_self_training)
print("TNR:", tnr_self_training)
print("FPR:", fpr_self_training)
print("FNR:", fnr_self_training)
print("F1-score:", f1_self_training)
print("AUC:", auc_self_training)


60/60 [==============================] - 0s 4ms/step
Self-Training Method
Accuracy: 0.6388888888888888
Precision: 0.16469428007889547
Recall: 0.2853075170842825
TNR: 0.7097989949748744
FPR: 0.29020100502512564
FNR: 0.7146924829157175
F1-score: 0.20883701542309296
AUC: 0.49755325602957845
